In [168]:
import gpt as g
import sys, os
import numpy as np

from scipy.linalg import expm
import time
import matplotlib.pyplot as plt
from gpt.qcd.gauge.smear import local_stout  
from gpt.ad import reverse as rad
from gpt.qcd.gauge.smear.differentiable import dft_diffeomorphism
import time

In [169]:
def trace_U(U):                                                      
    return sum(v for v in sum(u[:].real for u in g.eval(g.trace(U)))) / (4 * size**4 ) / 3.

In [170]:
num_steps = 1
def ftg(U, eps):
    global num_steps

    ##### dmuAmu ##############
    #B = U[0] -  g.adj(g.cshift(U[0], 0, -1))
    B = U[0] -  g.cshift(U[0], 0, -1)
    for mu in [1,2,3,]:
        #B += U[mu] -  g.adj(g.cshift(U[mu], mu, -1))
        B += U[mu] -  g.cshift(U[mu], mu, -1)

   #### masks for all even/odd sites ########
    """
    grid_cb = grid.checkerboarded(g.redblack)
    one_cb = g.complex(grid_cb)
    one_cb[:] = 1

    masks = {}
    for p in [g.even, g.odd]:
        m = g.complex(grid)
        m[:] = 0
        one_cb.checkerboard(p)
        g.set_checkerboard(m, one_cb)
        masks[p] = m
    
    if num_steps // 2 == 0:
        mask, imask = masks[g.odd], masks[g.odd.inv()] 
    else:
        mask, imask = masks[g.even], masks[g.even.inv()]
    
    num_steps += 1
    fm = g(mask + 1e-15 * imask)
    #fm = mask + 1e-15 * imask
    
    ###### apply masks ########
    #B *= fm # causes issues with action log det routine
    #B = g(B*fm)
    """
    
    # apply gtf 
    U_prime = []
    for mu in [0,1,2,3]:
        U_mu_prime = g(
                g.matrix.exp(  - eps * g.qcd.gauge.project.traceless_anti_hermitian(B) )  
                * U[mu] * g.matrix.exp(  + eps * g.qcd.gauge.project.traceless_anti_hermitian( g.cshift(B, mu, +1) ) ) 
        )
        U_prime.append(U_mu_prime)
    
    return U_prime


In [171]:
size = 4
grid = g.grid([size, size, size, size], g.double)
rng = g.random("t")

U = g.qcd.gauge.unit(grid)
rng.normal_element(U)


GPT :   10241.426504 s : Initializing gpt.random(t,vectorized_ranlux24_389_64) took 0.00103211 s


[lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double),
 lattice(ot_matrix_su_n_fundamental_group(3),double)]

In [172]:
def ft0(U):
    return ftg(U, eps=1e-2)
    
#dft = dft_diffeomorphism(U, ft0)
fr = g.algorithms.optimize.fletcher_reeves
ls2 = g.algorithms.optimize.line_search_quadratic

dft = g.qcd.gauge.smear.differentiable_field_transformation(
    Vgf,
    ft0,
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    #g.algorithms.inverter.fgmres(eps=1e-15, maxiter=30, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.inverter.fgcr(eps=1e-13, maxiter=100, restartlen=10),
    g.algorithms.optimize.non_linear_cg(
        maxiter=100, eps=1e-15, step=1e-1, line_search=ls2, beta=fr
    ),
)

dfm = dft.diffeomorphism()
ald = dft.action_log_det_jacobian()

In [180]:
# FT

Vgf = dft.inverse(U)

# conjugate momenta
U_mom = g.group.cartesian(Vgf)
#U_mom = g.group.cartesian(U)

action_gauge_mom = g.qcd.scalar.action.mass_term()
action_gauge = g.qcd.gauge.action.wilson(10.0)

metro = g.algorithms.markov.metropolis(rng)
inv = g.algorithms.inverter
sympl = g.algorithms.integrator.symplectic

log = sympl.log()

pure_gauge = True

"""
def transform(aa, s, i):
    aa_transformed = aa[i].transformed(s, indices=list(range(len(U))))
    aa_orig = aa[i]
    def draw(fields, rng, *extra):
        sfields = s(fields[0:len(U)]) + fields[len(U):]
        x = aa_orig.draw(sfields, rng, *extra)
        return x
    aa_transformed.draw = draw
    aa[i] = aa_transformed
"""

a_log_det = dft.action_log_det_jacobian()

GPT :   10362.074470 s : non_linear_cg: max_abs_step adjustment for step = 1.0742806226843562
GPT :   10362.097873 s : non_linear_cg: iteration 0: f(x) = 2.839411762004338e-02, |df|/sqrt(dof) = 1.233153e-02, beta = 0, step = 1.0
GPT :   10362.264728 s : non_linear_cg: max_abs_step adjustment for step = 1.0531526645282077
GPT :   10362.433378 s : non_linear_cg: max_abs_step adjustment for step = 1.0400573577841803
GPT :   10362.609456 s : non_linear_cg: max_abs_step adjustment for step = 1.0664580205332272
GPT :   10362.779510 s : non_linear_cg: max_abs_step adjustment for step = 1.0447379731107762
GPT :   10362.948823 s : non_linear_cg: max_abs_step adjustment for step = 1.0610817765955545
GPT :   10363.124234 s : non_linear_cg: max_abs_step adjustment for step = 1.0527504784253985
GPT :   10363.290511 s : non_linear_cg: max_abs_step adjustment for step = 1.055625497604385
GPT :   10363.455694 s : non_linear_cg: max_abs_step adjustment for step = 1.0553538977238326
GPT :   10363.618929

In [206]:
mom = [g.group.cartesian(v) for v in Vgf]
mom_prime = g.copy(mom)
rng.normal_element(mom_prime)

mom2 = dfm.jacobian(Vgf, U, mom_prime) # this appears to have fixed it???
ald = a_log_det(Vgf + mom2) # don't know if this is the correct log det!!!
g.message("Action log det jac:", ald)

GPT :   10649.306284 s : fgcr: converged in 11 iterations;  computed squared residual 3.382397e-25 / 2.138927e-23;  true squared residual 3.385119e-25 / 2.138927e-23
GPT :   10650.083486 s : fgcr: converged in 11 iterations;  computed squared residual 3.189350e-25 / 2.148702e-23;  true squared residual 3.190058e-25 / 2.148702e-23
GPT :   10650.084354 s : Action log det jac: 2148.7017788259063


In [207]:
def hamiltonian(draw):
    global mom2
    if draw:
        rng.normal_element(U_mom)
        
        mom = [g.group.cartesian(v) for v in Vgf]
        #mom = [g.group.cartesian(u) for u in U]
        mom_prime = g.copy(mom)
        rng.normal_element(mom_prime)

        W = dfm(Vgf)
        mom2 = dfm.jacobian(Vgf, W, mom_prime) # this appears to have fixed it???
        ald = a_log_det(Vgf + mom2)
        
        
        #s = action_gauge(U)

        
        s = action_gauge(Vgf)

        g.message("Gluonic action", s)
        g.message("Log-det action", ald)
        
        h = s + action_gauge_mom(U_mom) + ald
        #h = s + action_gauge_mom(U_mom)

    else:
        
        ald = a_log_det(Vgf + mom2)
        s = action_gauge(Vgf)
        #s = action_gauge(U)
        
        h = s + action_gauge_mom(U_mom) + ald
    return h, s


In [208]:
hamiltonian(True)

GPT :   10659.417935 s : fgcr: converged in 11 iterations;  computed squared residual 4.330220e-25 / 4.086285e-23;  true squared residual 4.330841e-25 / 4.086285e-23
GPT :   10660.208348 s : fgcr: converged in 11 iterations;  computed squared residual 4.913322e-25 / 4.108344e-23;  true squared residual 4.917362e-25 / 4.108344e-23
GPT :   10660.209273 s : Gluonic action 5919.303500651335
GPT :   10660.209477 s : Log-det action 4108.344470836949


(np.float64(14112.883337190156), np.float64(5919.303500651335))

In [272]:
def log_det_force():
    g.message("Compute log_det force")
    x = log(lambda: a_log_det.gradient(Vgf + mom2, Vgf), "log det")()
    g.message("second level force complete")
    return x

def sum_gauge_force():
    a =  a_log_det.gradient(Vgf + mom2, Vgf)
    b = action_gauge.gradient(Vgf, Vgf)

    return [a[i] + b[i] for i in [0,1,2,3,] ]

def gauge_force():
    #g.message("Compute gauge force")
    #x = log(lambda: action_gauge.gradient(Vgf, Vgf)[0] + a_log_det.gradient(Vgf + mom2, Vgf), "gauge")()
    x = log(lambda: action_gauge.gradient(Vgf, Vgf), "gauge")()
    #x = log(lambda: sum_gauge_force(), "gauge")()
    #g.message("first level force complete")
    return x
    
#def gauge_force222():
#    return gauge_force() + log_det_force()

In [258]:
a = action_gauge.gradient(Vgf, Vgf)
b = a_log_det.gradient(Vgf + mom2, Vgf)

GPT :   11763.888135 s : fgcr: converged in 11 iterations;  computed squared residual 2.119431e-25 / 4.044967e-23;  true squared residual 2.121322e-25 / 4.044967e-23
GPT :   11764.677780 s : fgcr: converged in 11 iterations;  computed squared residual 2.462337e-25 / 4.059022e-23;  true squared residual 2.464540e-25 / 4.059022e-23


In [276]:
iq = sympl.update_q(
    Vgf, log(lambda: action_gauge_mom.gradient(U_mom, U_mom), "gauge_mom")
)

ip_gauge = sympl.update_p(U_mom, gauge_force)

ip_log_det = sympl.update_p(U_mom, log_det_force)

# integrator
mdint = sympl.leap_frog(1, ip_log_det, sympl.leap_frog(1, ip_gauge, iq))


g.message(f"Integration scheme:\n{mdint}")

tau = 1.0
nsteps = 10



GPT :   12041.144413 s : Integration scheme:
                       : leap_frog(1, P, leap_frog(1, P, Q))
                       :   P(0.5, 0)
                       :   P(0.5, 0)
                       :   Q(1.0, 0)
                       :   P(0.5, 0)
                       :   P(0.5, 0)


In [277]:
no_accept_reject = False

def hmc(tau):
    accrej = metro(Vgf)
    #accrej = metro(U)
    g.message("After metro")

    h0, s0 = hamiltonian(True)
    
    #its0 = nsteps - 1
    nsteps = 10
    for i in range(nsteps):
        mdint(tau/nsteps)

    g.message("After mdint(tau)")
    h1, s1 = hamiltonian(False)
    g.message("After H(false)")
    
    if no_accept_reject:
        return [True, s1 - s0, h1 - h0]
    else:
        return [accrej(h1, h0), s1 - s0, h1 - h0]
   

In [278]:
hmc(1.)

GPT :   12049.029527 s : After metro
GPT :   12050.638182 s : fgcr: converged in 11 iterations;  computed squared residual 2.483168e-25 / 4.148038e-23;  true squared residual 2.482499e-25 / 4.148038e-23
GPT :   12051.415603 s : fgcr: converged in 11 iterations;  computed squared residual 3.185533e-25 / 4.163589e-23;  true squared residual 3.189002e-25 / 4.163589e-23
GPT :   12051.416742 s : Gluonic action 5201.658384291473
GPT :   12051.416920 s : Log-det action 4163.589077105746
GPT :   12051.417708 s : Compute log_det force
GPT :   12052.890291 s : fgcr: converged in 11 iterations;  computed squared residual 2.483168e-25 / 4.148038e-23;  true squared residual 2.482499e-25 / 4.148038e-23
GPT :   12053.678861 s : fgcr: converged in 11 iterations;  computed squared residual 3.185533e-25 / 4.163589e-23;  true squared residual 3.189002e-25 / 4.163589e-23
GPT :   12054.325981 s : second level force complete
GPT :   12054.338696 s : Compute log_det force
GPT :   12055.853573 s : fgcr: conve

[True, np.float64(-496.13708355721246), np.float64(-6.892192886658449)]

In [ ]:
accept, total = 0, 0
for it in range(20):
    #pure_gauge = it < 10
    no_accept_reject = it < 10
    g.message(pure_gauge, no_accept_reject)

    a, dS, dH = hmc(tau)
    accept += a
    total += 1


    #Uft = U
    #for s in reversed(sm):
    #    Uft = s(Uft)
        
    plaq = g.qcd.gauge.plaquette(U)
    #plaqft = g.qcd.gauge.plaquette(Uft)
    g.message(f"HMC plaq = {plaq}, dS = {dS}, dH = {dH}, acceptance = {accept/total}")
    g.message(f"Timing:\n{log.time}")
